# SSL-MAE / I-JEPA Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/SSL-MAE/blob/master/notebooks/colab_demo.ipynb)

Semi-supervised learning with masked autoencoders for remote sensing image classification.

## 1. Setup

In [ ]:
!git clone https://github.com/YOUR_USERNAME/SSL-MAE.git
%cd SSL-MAE
!pip install -q torch torchvision lightning timm omegaconf datasets scikit-learn iterstrat rich tqdm

In [ ]:
import os, sys, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightning as L

sys.path.insert(0, '.')

from src.utils.config import load_config
from src.utils.evaluate import evaluate, collect_targets
from src.utils.metrics import calculate_mlc_metrics, calculate_mcc_metrics
from src.models import ssl_mae, ssl_ijepa
from src.data.datamodule import SSLMAEDataModule
from src.trainers import MAELearner, IJEPALearner
from src.trainers.callbacks import EarlyStopping_, RichProgressBar_
from src.data.transforms import IMAGENET_MEAN, IMAGENET_STD

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': False, 'figure.dpi': 100,
    'axes.titleweight': 'bold',
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 2. Configuration

In [ ]:
MODEL_TYPE = "mae"                # "mae" or "ijepa"
FRACTION_LABELED = 0.1            # fraction of labeled data
EPOCHS = 20                       # reduce for quick demo
BATCH_SIZE = 16

# Modes to train and compare
MODES = ["semi_supervised", "supervised", "supervised_baseline"]

# Metrics to display
METRICS = [
    {"key": "average auprc",   "title": "Average AUPRC"},
    {"key": "ranking loss",    "title": "Ranking Loss"},
    {"key": "micro f1",        "title": "Micro F1"},
    {"key": "subset accuracy", "title": "Subset Accuracy"},
]

MODE_COLORS = {
    "semi_supervised": "#2176AE",
    "supervised": "#E8871E",
    "supervised_baseline": "#57A773",
}
MODE_LABELS = {
    "semi_supervised": "Semi-Supervised",
    "supervised": "Supervised",
    "supervised_baseline": "Baseline",
}

## 3. Train all modes

In [ ]:
results = {}  # mode → metrics DataFrame

for mode in MODES:
    print(f"\n{'='*50}")
    print(f"  Training: {MODEL_TYPE.upper()} — {MODE_LABELS[mode]}")
    print(f"{'='*50}\n")

    cfg = load_config(f"configs/{MODEL_TYPE}/ucm_mlc.yaml", cli_args=[
        f"training.mode={mode}",
        f"training.epochs={EPOCHS}",
        f"data.batch_size={BATCH_SIZE}",
        f"data.fraction_labeled={FRACTION_LABELED}",
        "data.num_workers=2",
        "training.devices=[0]",
    ])

    L.seed_everything(cfg.experiment.seed, workers=True)

    if MODEL_TYPE == "ijepa":
        model = ssl_ijepa(
            architecture=cfg.model.architecture,
            model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task,
            n_classes=cfg.data.n_classes,
            w=cfg.model.w,
            predictor_embed_dim=cfg.model.get('predictor_embed_dim', 384),
            predictor_depth=cfg.model.get('predictor_depth', 6),
            ema_decay=cfg.model.get('ema_target_decay', 0.996),
        )
        learner = IJEPALearner(model, cfg)
    else:
        model = ssl_mae(
            architecture=cfg.model.architecture,
            model_size=cfg.model.model_size,
            learning_task=cfg.data.learning_task,
            n_classes=cfg.data.n_classes,
            w=cfg.model.w,
        )
        learner = MAELearner(model, cfg)

    dm = SSLMAEDataModule(cfg)

    trainer = L.Trainer(
        max_epochs=EPOCHS, accelerator='auto', devices=[0],
        precision=cfg.training.precision, enable_checkpointing=False,
        logger=False,
        callbacks=[EarlyStopping_(metric='val_loss', mode='min', patience=cfg.training.patience),
                   RichProgressBar_()],
    )

    trainer.fit(learner, datamodule=dm)

    # Evaluate
    dm.setup()
    logits_list = trainer.predict(learner, datamodule=dm)
    logits = torch.cat(logits_list, dim=0).cpu().float()
    targets = collect_targets(dm.predict_dataloader())

    if cfg.data.learning_task == 'mlc':
        probas = torch.sigmoid(logits).numpy()
        preds = (probas > 0.5).astype(int)
        targets_np = targets.numpy()
        nonzero = targets_np.any(axis=1)
        Y = {'y_true': targets_np[nonzero], 'y_pred': preds[nonzero], 'y_scores': probas[nonzero]}
        df = calculate_mlc_metrics(Y)
    else:
        probas = torch.softmax(logits, dim=1).numpy()
        preds = probas.argmax(1).astype(int)
        targets_np = targets.numpy().astype(int)
        one_hot = np.eye(cfg.data.n_classes)[targets_np]
        Y = {'y_true': targets_np, 'y_pred': preds, 'y_scores': probas, 'one_hot': one_hot}
        df = calculate_mcc_metrics(Y)

    results[mode] = df
    print(f"\n{MODE_LABELS[mode]} metrics:")
    print(df)

print(f"\n{'='*50}")
print("  All training complete!")
print(f"{'='*50}")

## 4. Results comparison

In [ ]:
# Summary table
rows = []
for mode, df in results.items():
    row = {"Method": MODE_LABELS[mode]}
    for m in METRICS:
        if m["key"] in df.index:
            row[m["title"]] = f"{df.loc[m['key'], 'Metric Value']:.4f}"
    rows.append(row)

summary = pd.DataFrame(rows).set_index("Method")
display(summary)

## 5. Metric bar plots

In [ ]:
active_modes = [m for m in MODES if m in results]
n_metrics = len(METRICS)

fig, axes = plt.subplots(1, n_metrics, figsize=(3.5 * n_metrics, 3.5), sharey=False)
if n_metrics == 1: axes = [axes]

for i, m in enumerate(METRICS):
    ax = axes[i]
    vals, colors, labels = [], [], []
    for mode in active_modes:
        df = results[mode]
        if m["key"] in df.index:
            vals.append(df.loc[m["key"], "Metric Value"])
            colors.append(MODE_COLORS[mode])
            labels.append(MODE_LABELS[mode])

    if not vals:
        continue

    x = np.arange(len(vals))
    ax.bar(x, vals, color=colors, edgecolor='white', linewidth=0.5, width=0.6)

    # Auto-scale
    vmin, vmax = min(vals), max(vals)
    pad = max((vmax - vmin) * 0.25, 0.02)
    ax.set_ylim(max(0, vmin - pad), vmax + pad)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, rotation=25, ha='right')
    ax.set_title(m["title"])

fl_pct = int(FRACTION_LABELED * 100)
fig.suptitle(f"{MODEL_TYPE.upper()} — $f_l$={fl_pct}% — {EPOCHS} epochs", y=1.02)
plt.tight_layout()
plt.show()

## 6. Sample predictions

In [ ]:
def unnormalize(img_tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (img_tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

batch = next(iter(dm.predict_dataloader()))
N = min(6, batch[0].size(0))

fig, axes = plt.subplots(2, N, figsize=(2.5 * N, 5))
if N == 1: axes = axes[:, np.newaxis]

for i in range(N):
    img = unnormalize(batch[0][i])
    axes[0, i].imshow(img)
    axes[0, i].set_xticks([])
    axes[0, i].set_yticks([])
    if i == 0: axes[0, i].set_ylabel('Image')

    p = probas[i]
    top_k = np.argsort(p)[-5:][::-1]
    axes[1, i].barh(range(len(top_k)), p[top_k], color='#2176AE')
    axes[1, i].set_yticks(range(len(top_k)))
    axes[1, i].set_yticklabels([f'cls {k}' for k in top_k], fontsize=8)
    axes[1, i].set_xlim(0, 1)
    if i == 0: axes[1, i].set_ylabel('Top-5 probs')

plt.tight_layout()
plt.show()

## 7. Try different settings

Go back to **Section 2** and change:
- `MODEL_TYPE = "ijepa"` to try I-JEPA
- `FRACTION_LABELED = 0.01` for extreme low-label
- `EPOCHS = 50` for longer training

Then re-run from Section 3.